# Richardson iteration and conjugate gradients

How does the choice of search direction affect the number of steps needed to solve a finite element system? We compare Richardson, CG, and Jacobi-preconditioned CG on the same system, starting from $U^{(0)}=0$.

On the unit square $\Omega$ we seek $u_h$ in the continuous, piecewise affine space $V_h\subset H^1_0(\Omega)$ satisfying

$$
\begin{align*}
\int_\Omega \nabla u_h\cdot\nabla v_h\,dx &= \int_\Omega f v_h\,dx
\qquad\text{for every }v_h\in V_h,\\
f(x,y)&=2\bigl(x(1-x)+y(1-y)\bigr).
\end{align*}
$$

The mesh consists of equal squares split into right triangles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ngsolve import *
from ngsolve.webgui import Draw
from ngsolve.meshes import MakeStructured2DMesh

n = 12
mesh = MakeStructured2DMesh(quads=False, nx=n, ny=n)
Draw(mesh)
fes = H1(mesh, order=1, dirichlet=".*")
u, v = fes.TnT()

a = BilinearForm(fes, symmetric=True)
a += grad(u) * grad(v) * dx
a.Assemble()

source = 2 * (x * (1 - x) + y * (1 - y))
load = LinearForm(fes)
load += source * v * dx
load.Assemble();

## The system on the free vertices

NGSolve stores a coefficient for every vertex, including the prescribed boundary values. The projector $P$ sets boundary entries to zero. Applying it to both sides gives the free system $AU=b$ in vectors of the original length. On the range of $P$, the operator $A$ is symmetric positive definite. All iterations take place in this subspace.

In [ ]:
P = Projector(fes.FreeDofs(), True)
A = P @ a.mat @ P
b = (P * load.vec).Evaluate()
norm_b = Norm(b)
tol = 1e-8

print(f"{fes.FreeDofs().NumSet()} free vertex values")

## Richardson iteration

Each step moves along the residual with a fixed step length,

$$
\begin{align*}
r^{(k)} &= b-AU^{(k)}, &
U^{(k+1)} &= U^{(k)}+\omega r^{(k)}.
\end{align*}
$$

For this uniform mesh the interior stiffness stencil has diagonal $4$ and four nearest-neighbor entries $-1$ before boundary elimination. The eigenvalues of the free matrix lie strictly between $0$ and $8$, so $\omega=1/4$ satisfies $\omega\lambda_{\max}(A)<2$. We stop when the relative Euclidean residual $\|b-AU^{(k)}\|_2/\|b\|_2$ falls below the tolerance.

For the error $E^{(k)}=U_\ast-U^{(k)}$ this gives $E^{(k+1)}=(I-\omega A)E^{(k)}$. A component with eigenvalue $\lambda$ is multiplied by $1-\omega\lambda$. On this square grid the smallest and largest eigenvalues are $4(1-\cos(\pi/n))$ and $4(1+\cos(\pi/n))$. Their sum is eight, so the chapter's optimal constant step $2/(\lambda_{\min}+\lambda_{\max})$ is precisely $1/4$.

In [ ]:
omega = 0.25
U = b.CreateVector()
U[:] = 0
r = b.Copy()
richardson_residuals = [1.0]

for k in range(1, 2001):
    U.data += omega * r
    r.data = b - A * U
    richardson_residuals.append(Norm(r) / norm_b)
    if richardson_residuals[-1] <= tol:
        break
else:
    raise RuntimeError("Richardson reached its iteration limit")

print(f"Richardson took {k} steps. Relative residual {richardson_residuals[-1]:.2e}")

## Conjugate gradients

PCG uses the preconditioned residual $z=C^{-1}r$ and the scalar $\rho=r^\top z$. It chooses a step length $\tau$ that minimizes the energy along the current direction $d$. After the step, adding $(\rho_{\rm new}/\rho)d$ to the new $z$ gives the next conjugate direction. Setting $C^{-1}=I$ recovers CG.

The loop updates $r$ recursively for the search direction. To compare all methods with the same stopping criterion, we also compute $b-AU$ from the current iterate. This costs one extra matrix-vector product per step.

Why do successive directions help? After $k$ steps, ordinary CG minimizes $\|U_\ast-U\|_A$ over $U^{(0)}+\operatorname{span}\{r^{(0)},Ar^{(0)},\ldots,A^{k-1}r^{(0)}\}$ in exact arithmetic. Taking an $A$-orthogonal direction preserves the minimization already achieved along earlier directions.

In [ ]:
def pcg(A, b, C_inverse, tol, max_steps=200):
    U = b.CreateVector()
    U[:] = 0
    r = b.Copy()
    initial_norm = Norm(r)
    if initial_norm == 0:
        return U, [0.0]

    z = b.CreateVector()
    z.data = C_inverse * r
    d = z.Copy()
    rho = InnerProduct(r, z)
    residuals = [1.0]
    q = b.CreateVector()

    for k in range(max_steps):
        q.data = A * d
        tau = rho / InnerProduct(d, q)
        U.data += tau * d
        r.data -= tau * q

        residuals.append(Norm(b - A * U) / initial_norm)
        if residuals[-1] <= tol:
            return U, residuals

        z.data = C_inverse * r
        rho_new = InnerProduct(r, z)
        d = (z + (rho_new / rho) * d).Evaluate()
        rho = rho_new

    raise RuntimeError("PCG reached its iteration limit")

On the free vertices $P$ acts as the identity. For Jacobi-PCG, the inverse action divides each free residual entry by its diagonal stiffness entry. Applying NGSolve's point smoother as a matrix supplies precisely this diagonal scaling.

In [ ]:
jacobi = a.mat.CreateSmoother(fes.FreeDofs())
U_cg, cg_residuals = pcg(A, b, P, tol)
U_jacobi, jacobi_residuals = pcg(A, b, jacobi, tol)

print(f"CG took {len(cg_residuals) - 1} steps. Relative residual {cg_residuals[-1]:.2e}")
print(f"Jacobi-PCG took {len(jacobi_residuals) - 1} steps. Relative residual {jacobi_residuals[-1]:.2e}")

Here $\operatorname{diag}(A)=4I$ on the free vertices, so Jacobi scales every residual by the same factor. CG and Jacobi-PCG therefore produce identical iterates in exact arithmetic. The curves below show the first iterations. CG minimizes the energy error, and its Euclidean residual can increase during a step.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(richardson_residuals[:len(cg_residuals)], label="Richardson")
#ax.semilogy(richardson_residuals, label="Richardson")
ax.semilogy(cg_residuals, label="CG")
ax.semilogy(jacobi_residuals, "+", label="Jacobi-PCG")
ax.axhline(tol, color="0.5", linestyle=":", label="Tolerance")
ax.set(xlabel="Iteration", ylabel=r"$\|b-AU^{(k)}\|_2/\|b\|_2$")
ax.legend()
fig.tight_layout()
plt.show()

## Using NGSolve's CG solver

The built-in solver accepts the assembled stiffness matrix. The Jacobi inverse action zeroes the boundary entries of each correction, so the iterates stay in the free space. The solver monitors the preconditioned residual norm. We evaluate the true relative Euclidean residual of the returned vector to compare it with the value above.

In [ ]:
U_ngsolve = solvers.CG(mat=a.mat, rhs=b, pre=jacobi, tol=tol, printrates=True)
print(f"Relative residual of the NGSolve solution {Norm(b - A * U_ngsolve) / norm_b:.2e}")

## When does Jacobi preconditioning help?

On this mesh the free diagonal is $4I$. To isolate the effect of basis scaling, multiply the $i$-th free basis function by a positive factor $s_i$ between 1 and 20. The new matrix is $\widehat A=SAS$, where $S=\operatorname{diag}(s_i)$. The load becomes $Sb$ and the same finite element function has coefficients $S^{-1}U$. With $\widehat D=\operatorname{diag}(\widehat A)=4S^2$ we obtain

$$
\begin{align*}
\widehat D^{-1/2}\widehat A\,\widehat D^{-1/2}=A/4.
\end{align*}
$$

Compare the spectral condition numbers below. Jacobi removes the dependence on these artificial scale factors. The dependence on the mesh size remains.

In [ ]:
free = np.array(list(fes.FreeDofs()))
A_free = np.array(a.mat.ToDense())[free][:, free]
s = np.linspace(1.0, 20.0, len(A_free))
A_scaled = s[:, None] * A_free * s[None, :]
d = np.sqrt(np.diag(A_scaled))
A_jacobi = A_scaled / d[:, None] / d[None, :]
for name, matrix in [("Original basis", A_free), ("Rescaled basis", A_scaled),
                     ("Rescaled basis with Jacobi", A_jacobi)]:
    eigenvalues = np.linalg.eigvalsh(matrix)
    print(f"{name:27}  kappa = {eigenvalues[-1] / eigenvalues[0]:.2f}")

## What does a damped Jacobi step do to the error?

Using $D=\operatorname{diag}(A)$ in Richardson gives the update $U^{(k+1)}=U^{(k)}+\omega D^{-1}r^{(k)}$. Thus an error vector changes by $E\leftarrow E-\omega D^{-1}AE$. Here we can prescribe $E$ directly and watch the effect of four steps.

Take the nodal values of $\sin(j\pi x)\sin(j\pi y)$ on the same grid. The four-neighbor stencil gives the eigenvalue $1-\cos(j\pi/n)$ of $D^{-1}A$ for this mode. The energy norm after four steps should therefore be the fraction $|1-\omega(1-\cos(j\pi/n))|^4$ of its starting value. Which mode will remain when $\omega=2/3$?

In [ ]:
damping = 2 / 3
print("Mode     measured fraction    predicted fraction")
for j in (1, n - 1):
    mode = GridFunction(fes)
    mode.Interpolate(sin(j * np.pi * x) * sin(j * np.pi * y))
    E = (P * mode.vec).Evaluate()
    initial_energy = sqrt(InnerProduct(E, A * E))
    for step in range(4):
        correction = (jacobi * (A * E)).Evaluate()
        E.data -= damping * correction
    fraction = sqrt(InnerProduct(E, A * E)) / initial_energy
    predicted = abs(1 - damping * (1 - np.cos(j * np.pi / n)))**4
    print(f"{j:4d} {fraction:21.5f} {predicted:21.5f}")

A few damped Jacobi steps suppress the rapidly oscillating component while leaving much of the slowly varying one. This selective damping is called **smoothing**.

### Follow a mixed error for 1000 steps

To see the error along a line, take the one-dimensional Poisson problem with linear elements on 32 equal intervals and zero error at both endpoints. Start with the nodal values of $e^{(0)}(x)=\sin(\pi x)+0.45\sin(31\pi x)$. The first mode forms a single arch. The second changes sign between successive interior nodes.

For the one-dimensional stiffness matrix, $D^{-1}A$ has diagonal $1$ and off-diagonals $-1/2$. With $\omega=2/3$, a damped Jacobi step therefore updates each interior value using its two neighbors

$$
e_i^{(k+1)}=(1-\omega)e_i^{(k)}+\frac{\omega}{2}\bigl(e_{i-1}^{(k)}+e_{i+1}^{(k)}\bigr).
$$

The solid line joins the computed nodal errors. The dashed line follows just the first mode, whose amplitude is multiplied by $\mu_1=1-\omega(1-\cos(\pi/32))$ at each step. All six panels use the same vertical scale.

In [ ]:
n_intervals = 32
nodes = np.linspace(0, 1, n_intervals + 1)
slow_mode = np.sin(np.pi * nodes)
error = slow_mode + 0.45 * np.sin((n_intervals - 1) * np.pi * nodes)
error[[0, -1]] = 0
mu_slow = 1 - damping * (1 - np.cos(np.pi / n_intervals))

steps_to_plot = (0, 1, 4, 100, 500, 1000)
profiles = {0: error.copy()}
for k in range(1, steps_to_plot[-1] + 1):
    error[1:-1] = (1 - damping) * error[1:-1] + (damping / 2) * (error[:-2] + error[2:])
    if k in steps_to_plot:
        profiles[k] = error.copy()

fig, axes = plt.subplots(2, 3, figsize=(9, 4.8), sharex=True, sharey=True, layout="constrained")
for ax, k in zip(axes.flat, steps_to_plot):
    ax.plot(nodes, profiles[k], ".-", color="tab:blue", markersize=3, label="Jacobi error")
    ax.plot(nodes, mu_slow**k * slow_mode, "--", color="tab:orange", label="Slow mode")
    ax.set_title(f"$k={k}$")
    ax.set(xlim=(0, 1), ylim=(-0.06, 1.5), xticks=(0, 0.5, 1))
    ax.grid(alpha=0.2)
fig.supxlabel("$x$")
fig.supylabel("Nodal error")
axes[0, -1].legend(loc="upper right", fontsize=8)
plt.show()

After four steps the oscillation has less than $2\%$ of its initial amplitude, so the computed error nearly coincides with the smooth arch. The arch itself still has about $73\%$, $20\%$, and $4\%$ of its initial height after 100, 500, and 1000 steps. A smooth-looking iterate can therefore still have a substantial error.